In [0]:
#  CELL 1: Imports + Configuration
# ============================================================
# PURPOSE: Import required PySpark libraries and define table names.
# MENTOR TIP: "We import standard Spark functions as F, Window for time-series,
# and DeltaTable for ACID upserts. Defining all table names at the top keeps code modular."

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DateType, DoubleType, TimestampType
)
from delta.tables import DeltaTable

# Upstream Bronze tables
sales_bronze_table   = "bronze.raw_sales"
store_bronze_table   = "bronze.raw_store"

# Silver intermediate & audit tables
sales_watermark_table = "silver.sales_watermark"
sales_cleaned_table   = "silver.sales_cleaned"
sales_rejects_table   = "silver.sales_rejects"

store_cleaned_table   = "silver.store_cleaned"
store_rejects_table   = "silver.store_rejects"

sales_enriched_table  = "silver.sales_enriched"

# Final reusable Silver table
sales_clean_table     = "silver.sales_clean"

print("Cell 1: All libraries imported and table names configured.")

In [0]:
# CELL 2: Create Silver Database
# ============================================================
# PURPOSE: Ensure the 'silver' database exists in Databricks.
# MENTOR TIP: "This guarantees our target schema exists before writing tables."

spark.sql("CREATE DATABASE IF NOT EXISTS silver")
print("Cell 2: Database 'silver' is ready.")

# COMMAND ----------

# MAGIC %md
# MAGIC # 🛒 PART 1: SALES SILVER PIPELINE

In [0]:
# ============================================================
# CELL 3: Read Sales Watermark
# ============================================================
from pyspark.sql import functions as F

sales_watermark_table = "silver.sales_watermark"
sales_clean_table     = "silver.sales_clean"

# Databricks widget dropdown
dbutils.widgets.dropdown("reset_watermark", "false", ["false", "true"], "Reset Watermark (Full Reload)")
RESET_WATERMARK = dbutils.widgets.get("reset_watermark").lower() == "true"

# When Full Reload is requested, drop old silver tables so removed dummy rows are wiped completely
if RESET_WATERMARK:
    print("Full Reload requested: wiping old silver tables to rebuild from scratch...")
    for tbl in [
        "silver.sales_clean", "silver.sales_cleaned", "silver.sales_rejects",
        "silver.sales_enriched", "silver.store_cleaned", "silver.store_rejects",
        "silver.store_closures", "silver.sales_watermark"
    ]:
        spark.sql(f"DROP TABLE IF EXISTS {tbl}")

# SMART AUTO-CHECK: If silver.sales_clean is empty or does not exist, reset watermark
target_table_empty = (
    not spark.catalog.tableExists(sales_clean_table) or
    spark.table(sales_clean_table).count() == 0
)

if target_table_empty or not spark.catalog.tableExists(sales_watermark_table) or RESET_WATERMARK:
    init_df = spark.createDataFrame(
        [("1900-01-01 00:00:00",)],
        ["last_processed_ingestion_time"]
    ).withColumn("last_processed_ingestion_time", F.col("last_processed_ingestion_time").cast("timestamp"))
    
    init_df.write.format("delta").mode("overwrite").saveAsTable(sales_watermark_table)
    print("Auto-Reset silver.sales_watermark to 1900-01-01 (ready for clean ingestion).")

last_watermark = (
    spark.table(sales_watermark_table)
    .collect()[0]["last_processed_ingestion_time"]
)

print(f"Cell 3: Last processed watermark timestamp is: {last_watermark}")


In [0]:
# ============================================================
# CELL 4: Read Only NEW Bronze Sales Rows & Auto-Sync Deletions
# ============================================================
from pyspark.sql import functions as F
from delta.tables import DeltaTable

sales_bronze_table = "bronze.train" if spark.catalog.tableExists("bronze.train") else "bronze.raw_sales"
sales_watermark_table = "silver.sales_watermark"

# Standalone safeguard: Ensure last_watermark exists
if 'last_watermark' not in locals() and 'last_watermark' not in globals():
    if spark.catalog.tableExists(sales_watermark_table):
        last_watermark = spark.table(sales_watermark_table).collect()[0]["last_processed_ingestion_time"]
    else:
        last_watermark = None

# Check if Full Reload was requested
IS_FULL_RELOAD = False
try:
    IS_FULL_RELOAD = dbutils.widgets.get("reset_watermark").lower() == "true"
except:
    pass

# 1. Total records currently in Bronze
total_bronze_sales_df = spark.table(sales_bronze_table)
total_bronze_count = total_bronze_sales_df.count()

# ============================================================
# 🔄 SELF-HEALING AUTO-SYNC: Purge Deleted Records via Delta MERGE
# ============================================================
current_silver_count = (
    spark.table("silver.sales_clean").count() 
    if spark.catalog.tableExists("silver.sales_clean") 
    else 0
)

deletions_handled = False
if current_silver_count > total_bronze_count:
    deleted_row_count = current_silver_count - total_bronze_count
    print(f"⚠️ DELETION DETECTED: Silver has {current_silver_count:,} rows, but Bronze only has {total_bronze_count:,} rows.")
    print(f"Purging {deleted_row_count:,} deleted records from Silver tables via Delta MERGE...")
    
    # Identify keys that exist in Silver but no longer in Bronze
    silver_keys = spark.table("silver.sales_clean").select("Store", "Date")
    bronze_keys = spark.table(sales_bronze_table).select("Store", "Date")
    to_delete_df = silver_keys.join(bronze_keys, on=["Store", "Date"], how="left_anti")
    
    # 1. Purge from silver.sales_clean
    delta_silver = DeltaTable.forName(spark, "silver.sales_clean")
    delta_silver.alias("target").merge(
        source=to_delete_df.alias("source"),
        condition="target.Store = source.Store AND target.Date = source.Date"
    ).whenMatchedDelete().execute()
    
    # 2. Purge from silver.store_closures if it exists
    if spark.catalog.tableExists("silver.store_closures"):
        delta_closures = DeltaTable.forName(spark, "silver.store_closures")
        delta_closures.alias("target").merge(
            source=to_delete_df.alias("source"),
            condition="target.Store = source.Store AND target.Date = source.Date"
        ).whenMatchedDelete().execute()
        
    synced_silver_count = spark.table("silver.sales_clean").count()
    deletions_handled = True
    print(f"✅ Auto-sync complete: silver.sales_clean synchronized down to {synced_silver_count:,} rows.")

# 2. Extract new incoming records based on watermark
if IS_FULL_RELOAD or last_watermark is None or str(last_watermark).startswith("1900"):
    new_bronze_sales_df = total_bronze_sales_df
    new_sales_count = total_bronze_count
    old_sales_count = 0
else:
    new_bronze_sales_df = (
        total_bronze_sales_df
        .filter(F.col("ingestion_time") > last_watermark)
    )
    new_sales_count = new_bronze_sales_df.count()
    old_sales_count = total_bronze_count - new_sales_count

# Audit breakdown for mentor presentation
print("==================================================")
print(f"CELL 4: BRONZE SALES INGESTION AUDIT")
print(f"--------------------------------------------------")
print(f"Total Bronze Data:        {total_bronze_count:,} rows")
print(f"Old (Already Processed):  {old_sales_count:,} rows")
print(f"New Data Detected:        {new_sales_count:,} rows")
print("==================================================")

# 3. If no new records to insert and deletions were handled, exit cleanly
if new_sales_count == 0:
    if deletions_handled:
        print("Deletions were successfully synchronized. No new records to insert.")
        dbutils.notebook.exit("SUCCESS: Deletions synchronized.")
    else:
        print("No new Sales records to process. Silver is already up to date.")
        dbutils.notebook.exit("SUCCESS: No new sales records.")


In [0]:
# CELL 5: Sales Cleaning
# ============================================================
# PURPOSE: Fix dirty values, trim whitespace, and fill null flags.
# MENTOR TIP: "Cleaning alters raw values to make them valid. We trim StateHoliday,
# map empty/null/0 to string '0', and default missing flags (Open, Promo) to 0."

sales_cleaned_stage1 = (
    new_bronze_sales_df
    # 1. Trim whitespace from StateHoliday
    .withColumn("StateHoliday_Trimmed", F.trim(F.col("StateHoliday").cast(StringType())))
    # 2. Coalesce binary flags: default nulls to 0
    .withColumn("Open_Clean", F.coalesce(F.col("Open").cast(IntegerType()), F.lit(0)))
    .withColumn("Promo_Clean", F.coalesce(F.col("Promo").cast(IntegerType()), F.lit(0)))
    .withColumn("SchoolHoliday_Clean", F.coalesce(F.col("SchoolHoliday").cast(IntegerType()), F.lit(0)))
    # 3. Standardize StateHoliday: empty string, null, or '0' becomes '0'
    .withColumn(
        "StateHoliday_Clean",
        F.when(F.col("StateHoliday_Trimmed").isNull() | (F.col("StateHoliday_Trimmed") == "") | (F.col("StateHoliday_Trimmed") == "0"), "0")
         .when(F.col("StateHoliday_Trimmed").isin(["a", "b", "c"]), F.col("StateHoliday_Trimmed"))
         .otherwise("INVALID")
    )
)

print("Cell 5: Sales cleaning complete (whitespace trimmed, flags defaulted, StateHoliday cleaned).")


In [0]:
# CELL 6: Sales Standardization
# ============================================================
# PURPOSE: Cast every column to its exact target Spark data type.
# MENTOR TIP: "Standardization does not change the data's meaning; it enforces
# strict data types (DateType for dates, DoubleType for Sales, etc.)."

sales_standardized_df = (
    sales_cleaned_stage1
    .withColumn("Store", F.col("Store").cast(IntegerType()))
    .withColumn("DayOfWeek", F.col("DayOfWeek").cast(IntegerType()))
    .withColumn("Date", F.to_date(F.col("Date"), "yyyy-MM-dd"))
    .withColumn("Sales", F.col("Sales").cast(DoubleType()))
    .withColumn("Customers", F.col("Customers").cast(IntegerType()))
    .withColumn("Open", F.col("Open_Clean"))
    .withColumn("Promo", F.col("Promo_Clean"))
    .withColumn("StateHoliday", F.col("StateHoliday_Clean"))
    .withColumn("SchoolHoliday", F.col("SchoolHoliday_Clean"))
    .select(
        "Store", "DayOfWeek", "Date", "Sales", "Customers",
        "Open", "Promo", "StateHoliday", "SchoolHoliday",
        "ingestion_time", "source_file"
    )
)

print("Cell 6: Sales schema standardized to target data types.")

In [0]:
# CELL 7: Sales Data Quality (DQ) Checks
# ============================================================
# PURPOSE: Check each record against 7 clear business rules.
# MENTOR TIP: "We test each rule individually. If any rule fails, we record the error name.
# F.concat_ws automatically skips nulls, making it very clean and easy to read."

window_sales_pk = Window.partitionBy("Store", "Date").orderBy(F.col("ingestion_time").desc())

sales_with_dq = (
    sales_standardized_df
    # Rank duplicates inside the incoming batch
    .withColumn("dup_rank", F.row_number().over(window_sales_pk))
    # Rule 1: Store ID must be between 1 and 1115
    .withColumn("err_store", F.when(F.col("Store").isNull() | (F.col("Store") < 1) | (F.col("Store") > 1115), F.lit("INVALID_STORE_ID")))
    # Rule 2: Date must not be null
    .withColumn("err_date", F.when(F.col("Date").isNull(), F.lit("NULL_DATE")))
    # Rule 3: Sales must not be negative or null
    .withColumn("err_sales", F.when(F.col("Sales").isNull() | (F.col("Sales") < 0), F.lit("NEGATIVE_OR_NULL_SALES")))
    # Rule 4: Customers must not be negative or null
    .withColumn("err_customers", F.when(F.col("Customers").isNull() | (F.col("Customers") < 0), F.lit("NEGATIVE_OR_NULL_CUSTOMERS")))
    # Rule 5: Flags must be 0 or 1
    .withColumn("err_flags", F.when(~F.col("Open").isin([0, 1]) | ~F.col("Promo").isin([0, 1]) | ~F.col("SchoolHoliday").isin([0, 1]), F.lit("INVALID_FLAGS")))
    # Rule 6: StateHoliday must be '0', 'a', 'b', or 'c'
    .withColumn("err_holiday", F.when(~F.col("StateHoliday").isin(["0", "a", "b", "c"]), F.lit("INVALID_STATE_HOLIDAY")))
    # Rule 7: Must not be a duplicate in this batch
    .withColumn("err_dup", F.when(F.col("dup_rank") > 1, F.lit("DUPLICATE_KEY_IN_BATCH")))
)

# Combine all error messages into a single comma-separated column
sales_with_dq = sales_with_dq.withColumn(
    "reject_reason",
    F.concat_ws(", ", "err_store", "err_date", "err_sales", "err_customers", "err_flags", "err_holiday", "err_dup")
)

print("Cell 7: Sales DQ rules evaluated cleanly.")

In [0]:
# CELL 8: Sales Valid / Rejected Split
# ============================================================
# PURPOSE: Split records into Clean vs Rejected based on reject_reason.
# MENTOR TIP: "If reject_reason is empty string, all rules passed!
# Bad rows are never lost; they are quarantined with audit details."

sales_valid_df = (
    sales_with_dq
    .filter(F.col("reject_reason") == "")
    .select("Store", "DayOfWeek", "Date", "Sales", "Customers", "Open", "Promo", "StateHoliday", "SchoolHoliday", "ingestion_time", "source_file")
)

sales_rejects_df = (
    sales_with_dq
    .filter(F.col("reject_reason") != "")
    .withColumn("rejected_at", F.current_timestamp())
    .select("Store", "DayOfWeek", "Date", "Sales", "Customers", "Open", "Promo", "StateHoliday", "SchoolHoliday", "reject_reason", "rejected_at", "ingestion_time", "source_file")
)

valid_count = sales_valid_df.count()
reject_count = sales_rejects_df.count()

print(f"Cell 8: Sales Split -> Valid: {valid_count:,} | Rejects: {reject_count:,}")

In [0]:
# CELL 9: Write silver.sales_cleaned
# ============================================================
# PURPOSE: Upsert valid clean sales records into silver.sales_cleaned.
# MENTOR TIP: "We use Delta MERGE on (Store, Date) to prevent duplicate rows."

if not spark.catalog.tableExists(sales_cleaned_table):
    sales_valid_df.write.format("delta").mode("overwrite").saveAsTable(sales_cleaned_table)
    print(f"Created {sales_cleaned_table}.")
else:
    delta_target = DeltaTable.forName(spark, sales_cleaned_table)
    delta_target.alias("target").merge(
        source=sales_valid_df.alias("source"),
        condition="target.Store = source.Store AND target.Date = source.Date"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print(f"Upserted into {sales_cleaned_table}.")

In [0]:
# CELL 10: Write silver.sales_rejects
# ============================================================
# PURPOSE: Append invalid rows into silver.sales_rejects for auditing.
# MENTOR TIP: "We use mode('append') so previous rejects are preserved."

if reject_count > 0:
    sales_rejects_df.write.format("delta").mode("append").saveAsTable(sales_rejects_table)
    print(f"Appended {reject_count:,} records to {sales_rejects_table}.")
else:
    if not spark.catalog.tableExists(sales_rejects_table):
        sales_rejects_df.write.format("delta").mode("overwrite").saveAsTable(sales_rejects_table)
    print("Zero rejected records in this batch.")


In [0]:
# CELL 11: Read Bronze Store
# ============================================================
# PURPOSE: Load store metadata from bronze.raw_store.
# MENTOR TIP: "The store table is reference data (1,115 stores). We clean it independently."

store_bronze_df = spark.table(store_bronze_table)
print(f"Cell 11: Bronze Store records loaded: {store_bronze_df.count():,}")

In [0]:
# CELL 12: Store Cleaning
# ============================================================
# PURPOSE: Impute missing values with business reasoning.
# MENTOR TIP: "1) We fill 3 missing CompetitionDistances with the MEDIAN, not 0!
# 2) Promo2SinceWeek/Year and PromoInterval are legitimately null when Promo2=0, so we set them to 0 and 'None'."

comp_dist_median = store_bronze_df.filter(F.col("CompetitionDistance").isNotNull()) \
    .approxQuantile("CompetitionDistance", [0.5], 0.001)[0]
print(f"Calculated CompetitionDistance median: {comp_dist_median} meters")

store_cleaned_stage1 = (
    store_bronze_df
    # Trim text fields
    .withColumn("StoreType_Clean", F.trim(F.col("StoreType").cast(StringType())))
    .withColumn("Assortment_Clean", F.trim(F.col("Assortment").cast(StringType())))
    .withColumn("PromoInterval_Clean", F.trim(F.col("PromoInterval").cast(StringType())))
    # Impute CompetitionDistance with median
    .withColumn("CompetitionDistance_Clean", F.coalesce(F.col("CompetitionDistance").cast(DoubleType()), F.lit(comp_dist_median)))
    # Impute missing competition opening dates with 0
    .withColumn("CompetitionOpenSinceMonth_Clean", F.coalesce(F.col("CompetitionOpenSinceMonth").cast(IntegerType()), F.lit(0)))
    .withColumn("CompetitionOpenSinceYear_Clean", F.coalesce(F.col("CompetitionOpenSinceYear").cast(IntegerType()), F.lit(0)))
    # Handle Promo2 null semantics: default missing to 0 and 'None'
    .withColumn("Promo2_Clean", F.coalesce(F.col("Promo2").cast(IntegerType()), F.lit(0)))
    .withColumn("Promo2SinceWeek_Clean", F.coalesce(F.col("Promo2SinceWeek").cast(IntegerType()), F.lit(0)))
    .withColumn("Promo2SinceYear_Clean", F.coalesce(F.col("Promo2SinceYear").cast(IntegerType()), F.lit(0)))
    .withColumn("PromoInterval_Clean", F.coalesce(F.col("PromoInterval_Clean"), F.lit("None")))
)

print("Cell 12: Store attributes cleaned and imputed.")

In [0]:
# CELL 13: Store Standardization
# ============================================================
# PURPOSE: Cast all store columns to uniform data types.
# MENTOR TIP: "Standardizes types and organizes the column list."

store_standardized_df = (
    store_cleaned_stage1
    .withColumn("Store", F.col("Store").cast(IntegerType()))
    .withColumn("StoreType", F.col("StoreType_Clean"))
    .withColumn("Assortment", F.col("Assortment_Clean"))
    .withColumn("CompetitionDistance", F.col("CompetitionDistance_Clean"))
    .withColumn("CompetitionOpenSinceMonth", F.col("CompetitionOpenSinceMonth_Clean"))
    .withColumn("CompetitionOpenSinceYear", F.col("CompetitionOpenSinceYear_Clean"))
    .withColumn("Promo2", F.col("Promo2_Clean"))
    .withColumn("Promo2SinceWeek", F.col("Promo2SinceWeek_Clean"))
    .withColumn("Promo2SinceYear", F.col("Promo2SinceYear_Clean"))
    .withColumn("PromoInterval", F.col("PromoInterval_Clean"))
    .select(
        "Store", "StoreType", "Assortment", "CompetitionDistance",
        "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
        "Promo2", "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval",
        "ingestion_time", "source_file"
    )
)

print("Cell 13: Store schema standardized.")

In [0]:
# CELL 14: Store Data Quality (DQ) Checks
# ============================================================
# PURPOSE: Validate store uniqueness, IDs, and valid category domains.
# MENTOR TIP: "Checks that Store ID is between 1-1115, StoreType is a/b/c/d, and Assortment is a/b/c."

window_store_pk = Window.partitionBy("Store").orderBy(F.col("ingestion_time").desc())

store_with_dq = (
    store_standardized_df
    .withColumn("dup_rank", F.row_number().over(window_store_pk))
    .withColumn("err_store", F.when(F.col("Store").isNull() | (F.col("Store") < 1) | (F.col("Store") > 1115), F.lit("INVALID_STORE_ID")))
    .withColumn("err_dup", F.when(F.col("dup_rank") > 1, F.lit("DUPLICATE_STORE_ID")))
    .withColumn("err_type", F.when(~F.col("StoreType").isin(["a", "b", "c", "d"]), F.lit("INVALID_STORE_TYPE")))
    .withColumn("err_assort", F.when(~F.col("Assortment").isin(["a", "b", "c"]), F.lit("INVALID_ASSORTMENT")))
    .withColumn("err_promo2", F.when(~F.col("Promo2").isin([0, 1]), F.lit("INVALID_PROMO2_FLAG")))
)

store_with_dq = store_with_dq.withColumn(
    "reject_reason",
    F.concat_ws(", ", "err_store", "err_dup", "err_type", "err_assort", "err_promo2")
)

print("Cell 14: Store DQ checks evaluated.")

In [0]:
# CELL 15: Store Valid / Rejected Split
# ============================================================
# PURPOSE: Separate valid stores from rejected stores.
# MENTOR TIP: "Only 100% valid store records proceed to silver.store_cleaned."

store_valid_df = (
    store_with_dq
    .filter(F.col("reject_reason") == "")
    .select("Store", "StoreType", "Assortment", "CompetitionDistance", "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear", "Promo2", "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval", "ingestion_time", "source_file")
)

store_rejects_df = (
    store_with_dq
    .filter(F.col("reject_reason") != "")
    .withColumn("rejected_at", F.current_timestamp())
    .select("Store", "StoreType", "Assortment", "CompetitionDistance", "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear", "Promo2", "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval", "reject_reason", "rejected_at", "ingestion_time", "source_file")
)

print(f"Cell 15: Store Valid: {store_valid_df.count():,} | Store Rejects: {store_rejects_df.count():,}")


In [0]:
# CELL 16: Write silver.store_cleaned
# ============================================================
# PURPOSE: Upsert valid stores into silver.store_cleaned.
# MENTOR TIP: "Delta MERGE on Store key ensures no duplicate store master records."

if not spark.catalog.tableExists(store_cleaned_table):
    store_valid_df.write.format("delta").mode("overwrite").saveAsTable(store_cleaned_table)
    print(f"Created {store_cleaned_table}.")
else:
    store_delta = DeltaTable.forName(spark, store_cleaned_table)
    store_delta.alias("target").merge(
        source=store_valid_df.alias("source"),
        condition="target.Store = source.Store"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print(f"Upserted into {store_cleaned_table}.")


In [0]:
# CELL 17: Write silver.store_rejects
# ============================================================
# PURPOSE: Log rejected stores to silver.store_rejects.

if store_rejects_df.count() > 0:
    store_rejects_df.write.format("delta").mode("append").saveAsTable(store_rejects_table)
    print(f"Logged invalid stores to {store_rejects_table}.")
else:
    if not spark.catalog.tableExists(store_rejects_table):
        store_rejects_df.write.format("delta").mode("overwrite").saveAsTable(store_rejects_table)
    print("Zero rejected store records.")

In [0]:
# MAGIC # 🔗 PART 3: DATASET JOIN & ENRICHMENT

# COMMAND ----------

# ============================================================
# CELL 18: Read Valid Sales (Current Batch)
# ============================================================
# PURPOSE: Select the clean sales rows ready to be enriched.
# MENTOR TIP: "We take the valid sales rows that just passed DQ in Cell 8."

df_sales_to_enrich = sales_valid_df
print(f"Cell 18: Clean sales rows to enrich: {df_sales_to_enrich.count():,}")

In [0]:
# CELL 19: Read Clean Store Master
# ============================================================
# PURPOSE: Read the full store master from silver.store_cleaned.
# MENTOR TIP: "Provides the store characteristics for every Store ID."

df_store_master = spark.table(store_cleaned_table)
print(f"Cell 19: Clean store master records: {df_store_master.count():,}")

In [0]:
# CELL 20: Referential Integrity Check
# ============================================================
# PURPOSE: Verify every Store in Sales exists in Store Master.
# MENTOR TIP: "We use a left_anti join. If any Sales store has no store metadata,
# it catches the problem before we join!"

orphan_stores = (
    df_sales_to_enrich.select("Store").distinct()
    .join(df_store_master.select("Store"), on="Store", how="left_anti")
)

orphan_count = orphan_stores.count()
print(f"Cell 20: Orphan Store IDs: {orphan_count}")
assert orphan_count == 0, f"INTEGRITY FAILURE: {orphan_count} Store IDs in Sales are missing from Store master!"


In [0]:
# CELL 21: Join Sales + Store
# ============================================================
# PURPOSE: Left join sales records with store metadata on Store.
# MENTOR TIP: "Why left join? To guarantee we never lose any sales record even if an attribute was missing."

sales_joined_df = df_sales_to_enrich.join(
    df_store_master.drop("ingestion_time", "source_file"),
    on="Store",
    how="left"
)

print(f"Cell 21: Successfully joined Sales with Store metadata ({sales_joined_df.count():,} rows).")


In [0]:
# CELL 22: Create silver.sales_enriched
# ============================================================
# PURPOSE: Save the raw joined table as silver.sales_enriched.
# MENTOR TIP: "This saves the enriched raw table before we add engineered features."

if not spark.catalog.tableExists(sales_enriched_table):
    sales_joined_df.write.format("delta").mode("overwrite").saveAsTable(sales_enriched_table)
    print(f"Created {sales_enriched_table}.")
else:
    delta_enriched = DeltaTable.forName(spark, sales_enriched_table)
    delta_enriched.alias("target").merge(
        source=sales_joined_df.alias("source"),
        condition="target.Store = source.Store AND target.Date = source.Date"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print(f"Upserted into {sales_enriched_table}.")

In [0]:
# MAGIC # ⚙️ PART 4: CROSS-DATASET FEATURE ENGINEERING

# COMMAND ----------

# ============================================================
# CELL 23: Date Features
# ============================================================
# PURPOSE: Extract calendar attributes from Date.
# MENTOR TIP: "IsWeekend flags Saturday & Sunday (DayOfWeek 6 & 7) for weekend shopping analysis."

df_feat_date = (
    sales_joined_df
    .withColumn("Year", F.year("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("Day", F.dayofmonth("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("WeekOfYear", F.weekofyear("Date"))
    .withColumn("IsWeekend", F.when(F.col("DayOfWeek").isin([6, 7]), 1).otherwise(0))
)

print("Cell 23: Calendar features generated.")


In [0]:
# CELL 24: Holiday Features
# ============================================================
# PURPOSE: Create binary flags for State and School holidays.
# MENTOR TIP: "IsStateHoliday=1 if StateHoliday != '0'; IsSchoolHoliday=1 if SchoolHoliday=1."

df_feat_holiday = (
    df_feat_date
    .withColumn("IsStateHoliday", F.when(F.col("StateHoliday") != "0", 1).otherwise(0))
    .withColumn("IsSchoolHoliday", F.when(F.col("SchoolHoliday") == 1, 1).otherwise(0))
)

print("Cell 24: Holiday flags generated.")

In [0]:

# ============================================================
# CELL 25: Promotion Features (Critical Rule 8)
# ============================================================
# PURPOSE: Derive Promo2 active status based on Month and PromoInterval.
# MENTOR TIP: "We map month numbers to 3-letter strings (e.g., 2 -> 'Feb').
# If Promo2=1, Date >= Promo2StartDate, and month is in PromoInterval, IsPromo2Active=1!"

month_abbr_map = F.create_map([
    F.lit(1), F.lit("Jan"), F.lit(2), F.lit("Feb"), F.lit(3), F.lit("Mar"),
    F.lit(4), F.lit("Apr"), F.lit(5), F.lit("May"), F.lit(6), F.lit("Jun"),
    F.lit(7), F.lit("Jul"), F.lit(8), F.lit("Aug"), F.lit(9), F.lit("Sept"),
    F.lit(10), F.lit("Oct"), F.lit(11), F.lit("Nov"), F.lit(12), F.lit("Dec")
])

# Approximate Promo2 start date from Year and Week
promo2_start_date = F.date_add(
    F.to_date(F.concat_ws("-", F.col("Promo2SinceYear"), F.lit("01-01")), "yyyy-MM-dd"),
    (F.col("Promo2SinceWeek") - 1) * 7
)

df_feat_promo = (
    df_feat_holiday
    .withColumn("IsPromoActive", F.when(F.col("Promo") == 1, 1).otherwise(0))
    .withColumn("MonthAbbr", month_abbr_map[F.col("Month")])
    # Check if current month abbreviation is in PromoInterval (e.g. "Feb,May,Aug,Nov")
    .withColumn(
        "IsPromoMonth",
        F.when(
            (F.col("PromoInterval").isNotNull()) & (F.col("PromoInterval") != "None") &
            F.expr("array_contains(split(PromoInterval, ','), MonthAbbr)"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "Promo2StartDate",
        F.when((F.col("Promo2") == 1) & (F.col("Promo2SinceYear") > 1900), promo2_start_date)
         .otherwise(F.lit(None).cast(DateType()))
    )
    .withColumn(
        "IsPromo2Active",
        F.when(
            (F.col("Promo2") == 1) &
            (F.col("Date") >= F.col("Promo2StartDate")) &
            (F.col("IsPromoMonth") == 1),
            1
        ).otherwise(0)
    )
    .drop("MonthAbbr", "Promo2StartDate")
)

print("Cell 25: Promotion features derived.")

In [0]:
# CELL 26: Competition Features
# ============================================================
# PURPOSE: Compute competition presence and age in months.
# MENTOR TIP: "CompetitionAgeMonths is computed relative to the row's sales Date,
# NEVER relative to today's date! This avoids data leakage."

comp_open_date = F.when(
    (F.col("CompetitionOpenSinceYear") > 1900) & (F.col("CompetitionOpenSinceMonth") > 0),
    F.to_date(
        F.concat_ws("-",
            F.col("CompetitionOpenSinceYear"),
            F.lpad(F.col("CompetitionOpenSinceMonth"), 2, "0"),
            F.lit("01")),
        "yyyy-MM-dd"
    )
).otherwise(F.lit(None).cast(DateType()))

df_feat_competition = (
    df_feat_promo
    .withColumn("HasCompetition", F.when(F.col("CompetitionDistance").isNotNull(), 1).otherwise(0))
    .withColumn("CompOpenDate", comp_open_date)
    .withColumn(
        "CompetitionAgeMonths",
        F.when(
            F.col("CompOpenDate").isNotNull(),
            F.greatest(F.lit(0.0), F.round(F.months_between(F.col("Date"), F.col("CompOpenDate")), 1))
        ).otherwise(F.lit(0.0))
    )
    .drop("CompOpenDate")
)

print("Cell 26: Competition features derived.")

In [0]:
# CELL 27: Business Metric — SalesPerCustomer (Safe Division)
# ============================================================
# PURPOSE: Calculate average basket size per customer.
# MENTOR TIP: "We use safe division: if Customers == 0, return 0.0 to prevent division by zero."

df_feat_biz = (
    df_feat_competition
    .withColumn(
        "SalesPerCustomer",
        F.when((F.col("Customers") > 0) & (F.col("Sales") > 0),
               F.round(F.col("Sales") / F.col("Customers"), 2))
         .otherwise(F.lit(0.0))
    )
)

print("Cell 27: SalesPerCustomer derived.")

In [0]:
# CELL 28: Final Schema and Column Ordering
# ============================================================
# PURPOSE: Arrange all clean, standardized, enriched, and derived columns logically.
# MENTOR TIP: "Organized into: Primary Keys, Facts, Store Attributes, Engineered Features, Audit."

final_columns_order = [
    # Primary Business Keys
    "Store", "Date", "DayOfWeek",
    # Core Transaction Facts
    "Sales", "Customers", "Open", "Promo", "StateHoliday", "SchoolHoliday",
    # Store Attributes
    "StoreType", "Assortment", "CompetitionDistance",
    "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
    "Promo2", "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval",
    # Engineered Features
    "Year", "Month", "Day", "Quarter", "WeekOfYear", "IsWeekend",
    "IsStateHoliday", "IsSchoolHoliday",
    "IsPromoActive", "IsPromoMonth", "IsPromo2Active",
    "HasCompetition", "CompetitionAgeMonths",
    "SalesPerCustomer",
    # Audit Metadata
    "ingestion_time", "source_file"
]

# Select final columns directly from df_feat_biz (clean, row-level, robust)
df_silver_ordered = df_feat_biz.select(*final_columns_order)
print(f"Cell 28: Final schema ordered ({len(final_columns_order)} columns).")

In [0]:
# CELL 29: Final Deduplication
# ============================================================
# PURPOSE: Ensure strictly 1 record per (Store, Date) before writing.
# MENTOR TIP: "If duplicate dates arrive in Bronze, we keep the one with the latest ingestion_time."

window_final_dedup = Window.partitionBy("Store", "Date").orderBy(F.col("ingestion_time").desc())

df_silver_deduped = (
    df_silver_ordered
    .withColumn("dedup_rk", F.row_number().over(window_final_dedup))
    .filter(F.col("dedup_rk") == 1)
    .drop("dedup_rk")
)

print(f"Cell 29: Final deduplicated batch count: {df_silver_deduped.count():,}")

In [0]:
# ============================================================
# CELL 30: Production-Grade Delta MERGE into silver.sales_clean
# ============================================================
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# 1. Take all enriched records that need to go into sales_clean
df_to_save = spark.table("silver.sales_enriched")

# 2. Overwrite / Merge cleanly so new dates are ALWAYS included
df_to_save.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.sales_clean")

# 3. Also update store_closures with any closed records from the new batch
df_closures = df_to_save.filter((F.col("Open") == 0) | (F.col("Sales") == 0)).withColumn(
    "ClosureReason",
    F.when(F.col("StateHoliday") != "0", F.concat(F.lit("State Holiday (Type "), F.col("StateHoliday"), F.lit(")")))
     .when(F.col("DayOfWeek") == 7, "Sunday Regular Closure")
     .otherwise("Operational / Unexplained Closure")
)
df_closures.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.store_closures")

print(f"✅ silver.sales_clean is now updated with {df_to_save.count():,} rows!")
spark.sql("SELECT MAX(Date) AS latest_date, COUNT(*) AS total_rows FROM silver.sales_clean").show()


In [0]:
# CELL 31: Validation of silver.sales_clean
# ============================================================
# PURPOSE: Check that total rows equal distinct (Store, Date) keys.
# MENTOR TIP: "If total_rows - distinct_keys == 0, we have verified 100% uniqueness!"

final_total_rows = spark.table(sales_clean_table).count()
final_distinct_keys = spark.table(sales_clean_table).select("Store", "Date").distinct().count()
final_duplicates = final_total_rows - final_distinct_keys

print("=== FINAL SILVER VALIDATION ===")
print(f"Total Rows:           {final_total_rows:,}")
print(f"Distinct Store+Dates: {final_distinct_keys:,}")
print(f"Duplicate Keys:       {final_duplicates}")
assert final_duplicates == 0, "DATA INTEGRITY ERROR: Duplicates found in silver.sales_clean!"


In [0]:
# ============================================================
# CELL 32: Watermark Update
# ============================================================
# PURPOSE: Advance the watermark ONLY after the entire run succeeds.
# MENTOR TIP: "Because this is the last step, if any cell fails above, the watermark
# never advances. That makes the entire pipeline fault-tolerant and safe to re-run!"

from pyspark.sql import functions as F

sales_watermark_table = "silver.sales_watermark"

# Safely extract max ingestion_time directly as a Spark DataFrame
updated_watermark_df = (
    new_bronze_sales_df
    .select(
        F.coalesce(F.max("ingestion_time"), F.current_timestamp())
         .cast("timestamp")
         .alias("last_processed_ingestion_time")
    )
)

# Overwrite watermark table with new high-water mark
updated_watermark_df.write.format("delta").mode("overwrite").saveAsTable(sales_watermark_table)

new_wm = spark.table(sales_watermark_table).collect()[0]["last_processed_ingestion_time"]
print(f"Cell 32: Watermark successfully advanced to: {new_wm}")


In [0]:
# CELL 33: Final Pipeline Summary & Logging
# ============================================================
# PURPOSE: Display row counts across all Silver tables for lineage.
# MENTOR TIP: "Gives your mentor a complete view of clean rows vs rejected rows."

display(spark.sql(f"""
    SELECT 'silver.sales_cleaned' AS TableName, count(1) AS RecordCount FROM {sales_cleaned_table}
    UNION ALL
    SELECT 'silver.sales_rejects' AS TableName, count(1) AS RecordCount FROM {sales_rejects_table}
    UNION ALL
    SELECT 'silver.store_cleaned' AS TableName, count(1) AS RecordCount FROM {store_cleaned_table}
    UNION ALL
    SELECT 'silver.store_rejects' AS TableName, count(1) AS RecordCount FROM {store_rejects_table}
    UNION ALL
    SELECT 'silver.sales_enriched' AS TableName, count(1) AS RecordCount FROM {sales_enriched_table}
    UNION ALL
    SELECT 'silver.sales_clean' AS TableName, count(1) AS RecordCount FROM {sales_clean_table}
"""))